In [22]:
import pandas as pd
import numpy as np
from docplex.mp.model import Model
import math

# Sample data for trips
trip_data = pd.DataFrame({
    'trip_id': [1, 2, 3],
    'start_time': [480, 600, 900],
    'end_time': [540, 660, 960],
    'start_loc': ['A', 'B', 'C'],
    'end_loc': ['B', 'C', 'A'],
    'pax': [30, 25, 20]
})

locations = ['A', 'B', 'C', 'depot']
depot = 'depot'

# Constants
fixed_vehicle_cost = 100
fixed_depot_cost = 20
wait_cost_per_minute = 1
vehicle_capacity = 50
max_wait = 60

# Create graph nodes from trips
nodes = []
node_id = 0
trip_id_to_node_id = {}

for _, row in trip_data.iterrows():
    nodes.append({'id': node_id, 'type': 'start', 'trip_id': row['trip_id'], 'loc': row['start_loc'], 'time': row['start_time'], 'pax': row['pax']})
    start_node_id = node_id
    node_id += 1
    nodes.append({'id': node_id, 'type': 'end', 'trip_id': row['trip_id'], 'loc': row['end_loc'], 'time': row['end_time'], 'pax': row['pax']})
    end_node_id = node_id
    node_id += 1
    trip_id_to_node_id[row['trip_id']] = (start_node_id, end_node_id)

# Add depot start and end nodes
depot_start_node = {'id': node_id, 'type': 'depot_start', 'loc': depot, 'time': 0, 'pax': 0}
node_id += 1
depot_end_node = {'id': node_id, 'type': 'depot_end', 'loc': depot, 'time': 1440, 'pax': 0}
node_id += 1

nodes.extend([depot_start_node, depot_end_node])

V = nodes
node_id_lookup = {(v['loc'], v['time']): v['id'] for v in V}

# Create arcs
arcs = []

# Trip arcs
for trip_id, (start_id, end_id) in trip_id_to_node_id.items():
    arcs.append({'src': {'id': V[start_id]['id'], 'loc': V[start_id]['loc'], 'time': V[start_id]['time']},
                 'dst': {'id': V[end_id]['id'], 'loc': V[end_id]['loc'], 'time': V[end_id]['time']},
                 'type': 'trip', 'cost': 0, 'capacity': V[start_id]['pax']})

# Pull-out arcs
for v in V:
    if v['type'] == 'start':
        cost = abs(v['time'] - depot_start_node['time'])
        arcs.append({'src': {'id': depot_start_node['id'], 'loc': depot_start_node['loc'], 'time': depot_start_node['time']},
                     'dst': {'id': v['id'], 'loc': v['loc'], 'time': v['time']},
                     'type': 'pull-out', 'cost': cost + fixed_depot_cost, 'capacity': 150})

# Pull-in arcs
for v in V:
    if v['type'] == 'end':
        loc = v['loc']
        t = v['time']
        src_key = (loc, t)
        if src_key in node_id_lookup:
            arcs.append({
                'src': {'id': node_id_lookup[src_key], 'loc': loc, 'time': t},
                'dst': {'id': depot_end_node['id'], 'loc': depot_end_node['loc'], 'time': depot_end_node['time']},
                'type': 'pull-in',
                'cost': abs(depot_end_node['time'] - t) + fixed_depot_cost,
                'capacity': 150
            })

# Trip-to-trip chaining arcs
for i in V:
    if i['type'] == 'end':
        for j in V:
            if j['type'] == 'start' and i['loc'] == j['loc'] and 0 < j['time'] - i['time'] <= max_wait:
                cost = j['time'] - i['time']
                src_key = (i['loc'], i['time'])
                dst_key = (j['loc'], j['time'])
                if src_key in node_id_lookup and dst_key in node_id_lookup:
                    arcs.append({
                        'src': {'id': node_id_lookup[src_key], 'loc': i['loc'], 'time': i['time']},
                        'dst': {'id': node_id_lookup[dst_key], 'loc': j['loc'], 'time': j['time']},
                        'type': 'inventory',
                        'cost': cost * wait_cost_per_minute,
                        'capacity': 1
                    })

# Optimization model
mdl = Model("vehicle_routing")
K = list(range(len(trip_data)))
x = mdl.binary_var_dict(((arc['src']['id'], arc['dst']['id'], k) for arc in arcs for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, lb=0, name='T')

mdl.minimize(mdl.sum(arc['cost'] * x[(arc['src']['id'], arc['dst']['id'], k)] for arc in arcs for k in K))

# Constraints
for k in K:
    # Vehicle starts and ends at depot
    mdl.add_constraint(mdl.sum(x[(depot_start_node['id'], v['id'], k)] for v in V if v['type'] == 'start') == z[k])
    mdl.add_constraint(mdl.sum(x[(v['id'], depot_end_node['id'], k)] for v in V if v['type'] == 'end') == z[k])

    for v in V:
        if v['type'] in ['start', 'end']:
            incoming = mdl.sum(x[(u['id'], v['id'], k)] for u in V if (u['id'], v['id'], k) in x)
            outgoing = mdl.sum(x[(v['id'], w['id'], k)] for w in V if (v['id'], w['id'], k) in x)
            mdl.add_constraint(incoming == outgoing)

# Solve
mdl.solve(log_output=True)

# Output
for k in K:
    if z[k].solution_value > 0.5:
        print(f"Vehicle {k} is used.")
        for arc in arcs:
            if x.get((arc['src']['id'], arc['dst']['id'], k)) and x[(arc['src']['id'], arc['dst']['id'], k)].solution_value > 0.5:
                print(f"  {arc['type']} arc from {arc['src']} to {arc['dst']} with cost {arc['cost']}")


Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)

Root node processing (before b&c):
  Real time             =    0.00 sec. (0.00 ticks)
Parallel b&c, 32 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.00 sec. (0.00 ticks)


In [26]:
from docplex.mp.model import Model
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 220
MinUtilTime_2 = 150
MaxUtilTime = 1500

vehicle_fixed_cost = 1000
freq_penalty = 1000

nvehicle = 150
K = range(nvehicle)
H = range(HOURS)

bigM = 1440

Terminals=['ATB','HSK'] 
Depot='X'
freq1 = [0, 0, 0, 0, 3, 2, 3, 3, 3, 5, 4, 4, 4, 4, 3, 2, 3, 3, 3, 3, 3, 3, 0, 0]
freq2 = [0, 0, 0, 0, 4, 2, 3, 3, 3, 5, 4, 4, 4, 2, 3, 4, 4, 4, 3, 3, 3, 3, 0, 0]

arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {("ATB", "HSK"): 110, ("HSK", "ATB"): 110}
vehicle_fixed_cost = 1000
freq_penalty = 1000
fixed_depot_cost = 10000 
wait_cost_per_minute = 100
deadhead_cost_per_minute = 500 # Penalty to discourage depot pull-outs


# Parameters
# Depot = "X"
# Terminals = ["HSK", "ATB"]
# MINUTES_IN_DAY = 1440
# MinUtilTime_2 = 300
# H = list(range(24))  # 0 to 23 hours

# --- NODE CONSTRUCTION ---
nodes = []
node_id = 1
nodes.append({'id': node_id, 'time': 0, 'loc': Depot})
start_node_id = node_id
node_id += 1

for hour in H:
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': Terminals[0]})
        node_id += 1
    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': Terminals[1]})
        node_id += 1

nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': Depot})
end_node_id = node_id
node_id += 1

nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

# --- ARC CONSTRUCTION ---
E = []
synthetic_id = node_id
synthetic_nodes = []

for node in nodes:
    loc = node['loc']
    dst_time = node['time']
    if loc in Terminals:
        cost, N_bus = arc_data[(Depot, loc)]
        if dst_time >= cost:
            E.append({
                'src': {'id': start_node_id, 'time': 0, 'loc': Depot},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })

for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time + travel_time[(curr_loc, transitions[curr_loc])] <= MinUtilTime_2:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > MINUTES_IN_DAY:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_nodes.append(dst)

        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1
        })

        curr_src_id = synthetic_id
        curr_time = next_time
        curr_loc = next_loc
        synthetic_id += 1
        total_time += cost

    if curr_time <= MINUTES_IN_DAY:
        cost_to_X, N_bus_to_X = arc_data[(curr_loc, Depot)]
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': Depot},
            'cost': cost_to_X,
            'N_bus': N_bus_to_X
        })

V.extend(synthetic_nodes)

# --- FILTER ONLY END TRIP NODES + DEPOT NODES ---
depot_nodes = [node for node in V if node['id'] in [start_node_id, end_node_id]]

trip_end_nodes = []
seen_ids = set()

for arc in E:
    if arc['N_bus'] == 1:
        dst_node = arc['dst']
        if dst_node['id'] not in seen_ids:
            trip_end_nodes.append(dst_node)
            seen_ids.add(dst_node['id'])

# Combine and deduplicate
final_nodes = {node['id']: node for node in (depot_nodes + trip_end_nodes)}
filtered_nodes = list(final_nodes.values())

# --- OPTIONAL: Display filtered nodes ---
for node in sorted(filtered_nodes, key=lambda x: x['time']):
    print(f"ID: {node['id']}, Time: {node['time']}, Location: {node['loc']}")
inventory_arcs = []
# waiting_arcs = []
pullin_arcs = []

# Create mapping for trips ending at (loc, time)
# end_node_map = {}  # (loc, time) → list of trip dicts
# for trip in trip_nodes:
#     end_loc = transitions[trip['loc']]
#     end_time = trip['time'] + travel_time[(trip['loc'], end_loc)]
#     end_node_map.setdefault((end_loc, end_time), []).append(trip)

# # --------- 1. INVENTORY ARCS (Depot Pull-out + Chaining) ---------
# for dst_trip in trip_nodes:
#     dst_start_loc = dst_trip['loc']
#     dst_start_time = dst_trip['time']

#     # --- Depot Pull-out ---
#     pullout_time, _ = arc_data.get((Depot, dst_start_loc), (bigM, 0))
#     if dst_start_time >= pullout_time:
#         inventory_arcs.append({
#             'src': depot_start_node,
#             'dst': dst_trip,
#             'type': 'inventory',
#             'cost': fixed_depot_cost + pullout_time,
#             'capacity': 1
#         })

#     # --- Chaining from other trip end nodes ---
#     for (src_loc, src_time), trips_at_node in end_node_map.items():
#         if src_time >= dst_start_time:
#             continue  # can't chain backward

#         # Deadhead or wait cost
#         deadhead_time, deadhead_cost = arc_data.get((src_loc, dst_start_loc), (bigM, bigM))
#         arrival_time = src_time + deadhead_time

#         if arrival_time <= dst_start_time:
#             wait_time = dst_start_time - arrival_time
#             wait_cost = wait_time * (wait_cost_per_min if src_loc != Depot else 0)

#             total_cost = deadhead_cost + wait_cost
#             # For each vehicle at this node, connect once
#             inventory_arcs.append({
#                 'src': {'loc': src_loc, 'time': src_time},  # generic node (abstract from trip ID)
#                 'dst': dst_trip,
#                 'type': 'inventory',
#                 'cost': total_cost,
#                 'capacity': 1
#             })
inventory_arcs = []

# --- 1. Depot Pull-out arcs ---
for dst_trip in trip_nodes:
    dst_loc = dst_trip['loc']
    dst_time = dst_trip['time']

    travel_time_from_depot, travel_cost_from_depot = arc_data.get((Depot, dst_loc), (bigM, bigM))

    if dst_time >= travel_time_from_depot:
        total_cost = fixed_depot_cost + travel_cost_from_depot
        inventory_arcs.append({
            'src': depot_start_node,
            'dst': dst_trip,
            'type': 'inventory',
            'cost': total_cost,
            'capacity': 1
        })

# --- 2. Inventory chaining: Only one nearest reachable trip from each trip end ---
for (src_loc, src_time), trips_ending_here in end_node_map.items():
    for src_trip in trips_ending_here:
        best_dst_trip = None
        min_start_time = float('inf')

        for dst_trip in trip_nodes:
            dst_loc = dst_trip['loc']
            dst_time = dst_trip['time']

            deadhead_time, deadhead_cost = arc_data.get((src_loc, dst_loc), (bigM, bigM))
            arrival_time = src_time + deadhead_time

            if arrival_time <= dst_time and dst_time < min_start_time:
                best_dst_trip = dst_trip
                min_start_time = dst_time

        if best_dst_trip:
            dst_loc = best_dst_trip['loc']
            dst_time = best_dst_trip['time']
            deadhead_time, deadhead_cost = arc_data.get((src_loc, dst_loc), (bigM, bigM))
            arrival_time = src_time + deadhead_time
            wait_time = dst_time - arrival_time
            wait_cost = wait_time * (wait_cost_per_min if src_loc != Depot else 0)

            inventory_arcs.append({
                'src': {'loc': src_loc, 'time': src_time},
                'dst': best_dst_trip,
                'type': 'inventory',
                'cost': deadhead_cost + wait_cost,
                'capacity': 1
            })


# --------- 2. WAITING ARCS (Location l, Time t → Time t+1) ---------
# # Create waiting arcs between successive times at the same location
# all_locations = set(n['loc'] for n in V)
# for loc in all_locations:
#     loc_nodes = sorted([n for n in V if n['loc'] == loc], key=lambda x: x['time'])
#     for i in range(len(loc_nodes) - 1):
#         n1, n2 = loc_nodes[i], loc_nodes[i + 1]
#         time_diff = n2['time'] - n1['time']
#         if time_diff > 0:
#             cost = 0 if loc == Depot else time_diff * wait_cost_per_min
#             waiting_arcs.append({
#                 'src': n1,
#                 'dst': n2,
#                 'type': 'wait',
#                 'cost': cost,
#                 'capacity': 150
            # })

# --------- 3. PULL-IN ARCS (Trip End → Depot End Node) ---------
for (loc, time), trips in end_node_map.items():
    travel_time_to_depot, travel_cost = arc_data.get((loc, Depot), (bigM, bigM))
    if time + travel_time_to_depot <= MINUTES_IN_DAY:
        pullin_arcs.append({
            'src': {'loc': loc, 'time': time},
            'dst': depot_end_node,
            'type': 'pull-in',
            'cost': travel_cost + fixed_depot_cost,
            'capacity': 150
        })

# ------- Combine all arcs if needed -------
# all_arcs = inventory_arcs + waiting_arcs + pullin_arcs
# def print_arc(arc):
#     src = arc['src']
#     dst = arc['dst']
#     src_str = f"(Loc: {src['loc']}, Time: {src['time']})"
#     dst_str = f"(Loc: {dst['loc']}, Time: {dst['time']})"
#     return f"{arc['type'].upper()} ARC | From {src_str} --> To {dst_str} | Cost: {arc['cost']} | Capacity: {arc['capacity']}"
# Combine inventory and pull-in arcs
all_inventory_arcs = inventory_arcs + pullin_arcs
def normalize_node(node, fallback_id='unknown'):
    return {
        'loc': node.get('loc'),
        'time': node.get('time'),
        'id': node.get('id', fallback_id)
    }

# Convert inventory + pull-in arcs to standardized structure
all_inventory_arcs_standardized = []
for arc in inventory_arcs + pullin_arcs:
    standardized_arc = {
        'src': normalize_node(arc['src'], 'src'),
        'dst': normalize_node(arc['dst'], 'dst'),
        'cost': arc['cost'],
        'N_bus': arc['capacity']
    }
    all_inventory_arcs_standardized.append(standardized_arc)

# Print all arcs in E-like format
def print_arc(arc):
    src = arc['src']
    dst = arc['dst']
    print(
        f"ARC | From (Loc: {src['loc']}, Time: {src['time']}, ID: {src['id']}) "
        f"--> To (Loc: {dst['loc']}, Time: {dst['time']}, ID: {dst['id']}) | "
        f"Cost: {arc['cost']} | N_bus: {arc['N_bus']}"
    )

print("\n--- ALL INVENTORY-RELATED ARCS ---")
for arc in all_inventory_arcs_standardized:
    print_arc(arc)


# # Optional: Print all arcs
# print("\n--- ALL INVENTORY-RELATED ARCS ---")
# for arc in all_inventory_arcs:
#     print(print_arc(arc))

# print("\n--- INVENTORY ARCS ---")
# for arc in inventory_arcs:
#     print(print_arc(arc))

# # print("\n--- WAITING ARCS ---")
# # for arc in waiting_arcs:
# #     print(print_arc(arc))

# print("\n--- PULL-IN ARCS ---")
# for arc in pullin_arcs:
#     print(print_arc(arc))"add ids in the inventory arcs

ID: 1, Time: 0, Location: X
ID: 122, Time: 350, Location: HSK
ID: 123, Time: 350, Location: ATB
ID: 124, Time: 365, Location: ATB
ID: 125, Time: 370, Location: HSK
ID: 126, Time: 380, Location: ATB
ID: 127, Time: 390, Location: HSK
ID: 128, Time: 395, Location: ATB
ID: 129, Time: 410, Location: HSK
ID: 130, Time: 410, Location: ATB
ID: 131, Time: 440, Location: HSK
ID: 132, Time: 440, Location: ATB
ID: 133, Time: 470, Location: HSK
ID: 134, Time: 470, Location: ATB
ID: 135, Time: 490, Location: HSK
ID: 136, Time: 490, Location: ATB
ID: 137, Time: 510, Location: HSK
ID: 138, Time: 510, Location: ATB
ID: 139, Time: 530, Location: HSK
ID: 140, Time: 530, Location: ATB
ID: 141, Time: 550, Location: HSK
ID: 142, Time: 550, Location: ATB
ID: 143, Time: 570, Location: HSK
ID: 144, Time: 570, Location: ATB
ID: 145, Time: 590, Location: HSK
ID: 146, Time: 590, Location: ATB
ID: 147, Time: 610, Location: HSK
ID: 148, Time: 610, Location: ATB
ID: 149, Time: 630, Location: HSK
ID: 150, Time: 630, 

In [44]:
from docplex.mp.model import Model
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 330
MinUtilTime_2 = 150
MaxUtilTime = 1000

vehicle_fixed_cost = 1000
freq_penalty = 1000

nvehicle = 150
K = range(nvehicle)
H = range(HOURS)

bigM = 1440

Terminals=['ATB','HSK'] 
Depot='X'
# freq1 = [0, 0, 0, 0, 3, 2, 3, 3, 3, 5, 4, 4, 4, 4, 3, 2, 3, 3, 3, 3, 3, 3, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 2, 3, 3, 3, 5, 4, 4, 4, 2, 3, 4, 4, 4, 3, 3, 3, 3, 0, 0]
freq1 = [0, 0, 0, 0, 1, 1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 1, 0, 0, 1, 1, 1, 1, 0, 0]
freq2 = [0, 0, 0, 0, 1, 1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 1, 0, 0, 1, 1, 1, 1, 0, 0]

arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {("ATB", "HSK"): 110, ("HSK", "ATB"): 110}
vehicle_fixed_cost = 1000
freq_penalty = 1000
fixed_depot_cost = 10000 
wait_cost_per_minute = 100
deadhead_cost_per_minute = 500 # Penalty to discourage depot pull-outs


# Parameters
# Depot = "X"
# Terminals = ["HSK", "ATB"]
# MINUTES_IN_DAY = 1440
# MinUtilTime_2 = 300
# H = list(range(24))  # 0 to 23 hours

# --- NODE CONSTRUCTION ---
nodes = []
node_id = 1
nodes.append({'id': node_id, 'time': 0, 'loc': Depot})
start_node_id = node_id
node_id += 1

for hour in H:
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': Terminals[0]})
        node_id += 1
    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': Terminals[1]})
        node_id += 1

nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': Depot})
end_node_id = node_id
node_id += 1

nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

# --- ARC CONSTRUCTION ---
E = []
synthetic_id = node_id
synthetic_nodes = []

for node in nodes:
    loc = node['loc']
    dst_time = node['time']
    if loc in Terminals:
        cost, N_bus = arc_data[(Depot, loc)]
        if dst_time >= cost:
            E.append({
                'src': {'id': start_node_id, 'time': 0, 'loc': Depot},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })

for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time + travel_time[(curr_loc, transitions[curr_loc])] <= MinUtilTime_2:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > MINUTES_IN_DAY:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_nodes.append(dst)

        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1
        })

        curr_src_id = synthetic_id
        curr_time = next_time
        curr_loc = next_loc
        synthetic_id += 1
        total_time += cost

    if curr_time <= MINUTES_IN_DAY:
        cost_to_X, N_bus_to_X = arc_data[(curr_loc, Depot)]
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': Depot},
            'cost': cost_to_X,
            'N_bus': N_bus_to_X
        })

V.extend(synthetic_nodes)# Step 1: Index nodes by (loc, time)
depot_nodes = [node for node in V if node['id'] in [start_node_id, end_node_id]]

trip_end_nodes = []
seen_ids = set()

for arc in E:
    if arc['N_bus'] == 1:
        dst_node = arc['dst']
        if dst_node['id'] not in seen_ids:
            trip_end_nodes.append(dst_node)
            seen_ids.add(dst_node['id'])

# Combine and deduplicate
final_nodes = {node['id']: node for node in (depot_nodes + trip_end_nodes)}
R = list(final_nodes.values())

# Replace filtered_nodes with R
filtered_nodes = R

# node_map = {(n['loc'], n['time']): n for n in filtered_nodes}
# trip_nodes = [n for n in filtered_nodes if n['id'] not in [start_node_id, end_node_id]]
# depot_start_node = next(n for n in filtered_nodes if n['id'] == start_node_id)
# depot_end_node = next(n for n in filtered_nodes if n['id'] == end_node_id)

I = []

# --- 1. Depot Pull-out Arcs ---
for dst in trip_nodes:
    travel_time, travel_cost = arc_data.get((Depot, dst['loc']), (bigM, bigM))
    if dst['time'] >= travel_time:
        total_cost = fixed_depot_cost + travel_cost
        I.append({
            'src': depot_start_node,
            'dst': dst,
            'cost': total_cost,
            'N_bus': 150
        })

# --- 2. Inventory chaining: For each trip end, find next reachable trip ---
for src in trip_nodes:
    best_dst = None
    best_arrival_time = float('inf')

    for dst in trip_nodes:
        if src['id'] == dst['id']:
            continue  # skip self
        deadhead_time, deadhead_cost = arc_data.get((src['loc'], dst['loc']), (bigM, bigM))
        arrival_time = src['time'] + deadhead_time
        if arrival_time <= dst['time'] and dst['time'] < best_arrival_time:
            best_dst = dst
            best_arrival_time = dst['time']

    if best_dst:
        deadhead_time, deadhead_cost = arc_data.get((src['loc'], best_dst['loc']), (bigM, bigM))
        arrival_time = src['time'] + deadhead_time
        wait_time = best_dst['time'] - arrival_time
        wait_cost = wait_time * (wait_cost_per_min if src['loc'] != Depot else 0)

        I.append({
            'src': src,
            'dst': best_dst,
            'cost': deadhead_cost + wait_cost,
            'N_bus': 1
        })

# --- 3. Pull-in arcs (trip end to depot) ---
for src in trip_nodes:
    travel_time, travel_cost = arc_data.get((src['loc'], Depot), (bigM, bigM))
    if src['time'] + travel_time <= MINUTES_IN_DAY:
        I.append({
            'src': src,
            'dst': depot_end_node,
            'cost': travel_cost + fixed_depot_cost,
            'N_bus': 150
        })

# --- Generate unique arc keys ---
unique_inventory_keys = set()

for e in I:
    arc_key = (
        e['src']['id'], e['src']['time'], e['src']['loc'],
        e['dst']['id'], e['dst']['time'], e['dst']['loc'],
        e['cost'], e['N_bus']
    )
print("\n--- FINAL INVENTORY ARCS (filtered_nodes only) ---")
for e in I:
    print({
        'src': e['src'],
        'dst': e['dst'],
        'cost': e['cost'],
        'N_bus': e['N_bus']
    })

mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 2

x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in I for k in K), name=Depot)
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

node_ids_in_arcs = set()
for e in I:
    node_ids_in_arcs.add(e['src']['id'])
    node_ids_in_arcs.add(e['dst']['id'])

arrival_time = mdl.continuous_var_dict(((k, nid) for k in K for nid in node_ids_in_arcs), name='arr', lb=0, ub=MINUTES_IN_DAY)
freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)

mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in I for k in K) 
   
)
#     mdl.sum(vehicle_fixed_cost * z[k] for k in K) 
# )

# mdl.minimize(
#     mdl.sum(z[k] for k in K) 

#     # mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)

for k in K:
    mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I))

    mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I if e['src']['id'] == start_node_id) == z[k])
    mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I if e['dst']['id'] == end_node_id) == z[k])

    for node in R:
        i = node['id']
        if i != start_node_id and i != end_node_id:
            mdl.add_constraint(
                mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I if e['src']['id'] == i) ==
                mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I if e['dst']['id'] == i)
            )

for e in I:
    if e['N_bus'] == 1:
        mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for k in K) <= 1)

# for k in K:
#     mdl.add_constraint(T[k] == mdl.sum(e['cost'] * x.get((e['src']['id'], e['dst']['id'], k), 0) for e in I if e['N_bus'] == 1))
#     mdl.add_constraint(T[k] >= MinUtilTime * z[k])
#     mdl.add_constraint(T[k] <= MaxUtilTime * z[k])


for h in H:
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for e in I if e['N_bus'] == 1 and e['src']['loc'] == Terminals[0] and (e['src']['time'] // 60) == h
            for k in K
        ) >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for e in I if e['N_bus'] == 1 and e['src']['loc'] == Terminals[1] and (e['src']['time'] // 60) == h
            for k in K
        ) >= freq2[h]
    )

# # CPLEX tuning
mdl.context.cplex_parameters.timelimit = 500
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.05
mdl.context.cplex_parameters.threads = 8
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 1
  

solution = mdl.solve(log_output=True)

if solution:
    print("Objective:", solution.objective_value)
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")
        current_node = start_node_id
        while current_node != end_node_id:
            next_arcs = [e for e in I if e['src']['id'] == current_node and x.get((e['src']['id'], e['dst']['id'], k), 0).solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break
            arc = next_arcs[0]
            src, dst = arc['src'], arc['dst']
            src_h, src_m = divmod(int(src['time']), 60)
            dst_h, dst_m = divmod(int(dst['time']), 60)
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")
            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")
print(mdl.number_of_variables)


--- FINAL INVENTORY ARCS (filtered_nodes only) ---
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 122, 'time': 350, 'loc': 'HSK'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 123, 'time': 350, 'loc': 'ATB'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 124, 'time': 365, 'loc': 'ATB'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 125, 'time': 370, 'loc': 'HSK'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 126, 'time': 380, 'loc': 'ATB'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 127, 'time': 390, 'loc': 'HSK'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 128, 'time': 395, 'loc': 'ATB'}, 'cost': 10150, 'N_bus': 150}
{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 129, 'time': 410, 'loc': 'HSK'}, 'cost': 10150, 'N_bus': 150}
{'sr